# 02 – Feature Engineering

**Project:** Area Feasibility Scoring Model  
**Goal:** Build a leakage-safe feature engineering pipeline and validate that each feature adds signal.

---
### Contents
1. Setup
2. Train/Test Split (before any feature computation)
3. Feature Engineering with `LeakageSafeFeaturizer`
4. Leakage Audit
5. Feature Distributions
6. Correlation with Target
7. Feature Summary

## 1 · Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

from data_loader import prepare_dataset
from features import (
    LeakageSafeFeaturizer,
    FEATURE_COLS,
    CLASSIFICATION_TARGET,
    assert_no_leakage,
    build_target,
)

RANDOM_STATE = 42
USER_BUDGET  = 300_000

In [ ]:
# Load area-level data (synthetic for demonstration)
area_df = prepare_dataset(user_budget=USER_BUDGET, use_synthetic=True)

# Build classification target BEFORE splitting (it's just a column comparison, no stats)
y = build_target(area_df)

print(f'Areas: {len(area_df)}')
print(f'Target distribution:\n{y.value_counts().rename({0: "Not Affordable", 1: "Affordable"})}')

## 2 · Train/Test Split (BEFORE feature computation)

> **Key leakage principle:** We split the data *before* fitting the featurizer.  
> The `LeakageSafeFeaturizer` is then `fit` **only on training rows** and
> `transform`ed on both train and test independently.

In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    area_df, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(f'Train: {len(X_train_raw)} rows  |  Test: {len(X_test_raw)} rows')
print(f'Train affordability rate : {y_train.mean():.1%}')
print(f'Test  affordability rate : {y_test.mean():.1%}')

## 3 · Feature Engineering with `LeakageSafeFeaturizer`

In [ ]:
featurizer = LeakageSafeFeaturizer(add_interactions=True)

# fit on TRAINING DATA ONLY
featurizer.fit(X_train_raw)

# transform both splits independently
X_train = featurizer.transform(X_train_raw)
X_test  = featurizer.transform(X_test_raw)

print('Training features shape:', X_train.shape)
print('Test features shape    :', X_test.shape)
X_train.head()

In [ ]:
X_train.describe().T

## 4 · Leakage Audit

In [ ]:
# Confirm no feature is also a target column
assert_no_leakage(FEATURE_COLS, [CLASSIFICATION_TARGET, 'median_price'])

# Confirm raw price columns are NOT in the engineered feature set
raw_price_cols = {'median_price', 'price_25th', 'price_75th', 'budget'}
leaky = raw_price_cols & set(X_train.columns)
if leaky:
    print(f'WARNING: raw price columns found in features: {leaky}')
else:
    print('✓ Raw price columns correctly excluded from feature matrix.')
    print('✓ Feature matrix columns:', list(X_train.columns))

## 5 · Feature Distributions

In [ ]:
n_cols = 4
n_rows = int(np.ceil(len(FEATURE_COLS) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3 * n_rows))
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLS):
    axes[i].hist(X_train[col].dropna(), bins=30, color='steelblue', edgecolor='white')
    axes[i].set_title(col, fontsize=10)

# Hide unused subplot slots
for j in range(len(FEATURE_COLS), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Training-Set Feature Distributions', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

## 6 · Correlation with Target

In [ ]:
# Point-biserial correlation between each feature and the binary target
correlations = (
    pd.concat([X_train, y_train.reset_index(drop=True)], axis=1)
    .corr()[CLASSIFICATION_TARGET]
    .drop(CLASSIFICATION_TARGET, errors='ignore')
    .sort_values(key=abs, ascending=False)
)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in correlations]
ax.barh(correlations.index[::-1], correlations.values[::-1], color=colors[::-1], edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Pearson Correlation with Target')
ax.set_title('Feature–Target Correlation')
plt.tight_layout()
plt.show()

print(correlations.to_string())

In [ ]:
# Feature–feature correlation heatmap (check for multicollinearity)
fig, ax = plt.subplots(figsize=(10, 8))
corr_matrix = X_train.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', linewidths=0.5, ax=ax,
    annot_kws={'size': 8}
)
ax.set_title('Feature–Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## 7 · Feature Summary

| Feature | Description | Expected signal |
|---|---|---|
| `affordability_ratio` | budget / median_price | High correlation with target |
| `budget_vs_p25` | budget / 25th-pctile price | Affordability vs cheaper end |
| `budget_vs_p75` | budget / 75th-pctile price | Affordability vs expensive end |
| `price_spread` | IQR of area prices | Market volatility |
| `price_spread_pct` | IQR / median_price | Relative volatility |
| `pct_within_budget` | Estimated % listings ≤ budget | Supply signal |
| `budget_deficit` | median_price − budget | Signed gap |
| `log_budget` | log1p(budget) | Scale normalisation |
| `log_median_price` | log1p(median_price) | Scale normalisation |
| `log_num_listings` | log1p(num_listings) | Property supply |
| `ratio_x_spread` | ratio × spread_pct | Interaction term |

**Next step:** → `03_modeling.ipynb`